# Additional evidence heatmaps — Evidence heatmaps for the differing loci (data processing)

For CR1, MS4A4A and CASS4: every variant in a SuSiE-RSS or SuSiE-rRSS credible set, with its PIP under each method, AD GWAS −log10 p, cV2F, xQTL inclusion score and colocalization VCP by cell type. Values are min–max scaled per row for the fill.

Note: this uses the newer `AD_locus_xQTL_summary_2026-09.xlsx` (as the original notebook did), while Fig 6D/6E use `unified_AD_loci_xQTL_summary.xlsx`.

**Output:** `figure_data/Exploratory.Differing_Loci_Evidence_Heatmap.rds` — named list; each = `list(plot_long, x_labels, title)`

In [ ]:
source("../../R/utils.R")
library(readxl)

In [ ]:
### Load AD GWAS explorer data
AD_loci <- suppressMessages(
  suppressWarnings(
    read_excel(
      "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_locus_xQTL_summary_2026-09.xlsx",
      skip = 2
    )
  )
)

cs_counts_df <- readRDS("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export//method_compare.cs_counts.rds")
region_diff  <- cs_counts_df %>% filter(diff != 0) %>% filter(block_id != apoe_block) %>% filter(!str_detect(study, "2026"))

In [ ]:
## ---- the two fits + AD_loci subset for one block/study ----
load_fits <- function(bid, sid, AD_loci,
                      root      = "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10",
                      susie_sub = "AD_GWAS_rss_baseline_r4_L10",
                      ebmix_sub = "AD_GWAS_rss_ebmix_r4_L10") {
  fname   <- paste0(sid, ".", bid, ".gwas_finemap.rds")
  get_fit <- function(f) attributes(readRDS(f))$elementMetadata$susieFit[[1]]
  m   <- str_match(bid, "^chr([0-9XY]+)_([0-9]+)_([0-9]+)$")
  reg <- list(chr = m[, 2], start = as.numeric(m[, 3]), end = as.numeric(m[, 4]))
  list(
    fit_ebmix = get_fit(file.path(root, ebmix_sub, "output/fine_mapping", fname)),
    fit_susie = get_fit(file.path(root, susie_sub, "output/fine_mapping", fname)),
    AD_subset = AD_loci %>% filter(Chr == as.integer(reg$chr), Pos >= reg$start, Pos <= reg$end),
    reg = reg, bid = bid, sid = sid
  )
}

## ---- variant x feature table for the heatmap ----
locus_heatmap_data <- function(loc) {
  fit_ebmix <- loc$fit_ebmix; fit_susie <- loc$fit_susie; AD_subset <- loc$AD_subset

  cs_vars <- function(fit) if (is.null(fit) || is.null(fit$sets$cs)) character(0)
                           else colnames(fit$alpha)[unlist(fit$sets$cs)]
  cs_union <- union(cs_vars(fit_ebmix), cs_vars(fit_susie))
  vpos <- as.integer(str_match(cs_union, "^chr[0-9]+[:_]([0-9]+)")[, 2])
  o <- order(vpos); cs_union <- cs_union[o]; vpos <- vpos[o]

  pipv_cs <- function(fit) {
    v <- colnames(fit$alpha)
    cs_v <- if (is.null(fit$sets$cs)) character(0) else v[unlist(fit$sets$cs)]
    val <- setNames(as.numeric(fit$pip), v)[cs_union]
    val[!(cs_union %in% cs_v)] <- NA
    val
  }
  vcp_col <- names(AD_subset)[str_detect(names(AD_subset), "^Colocalization VCP")][1]
  ad <- AD_subset %>%
    transmute(pos = Pos, gwas = `log10(pval)`, cv2f = `cV2F score`,
              incl = `maximum inclusion score`, ctx = Context, vcp = .data[[vcp_col]]) %>%
    mutate(ct = str_extract(ctx, "^[A-Za-z]+"),
           ct = ifelse(ct %in% c("Ast","Exc","Inh","Mic","Oli","OPC"), ct, "Bulk/other"))
  safe_max <- function(x) { x <- x[!is.na(x)]; if (length(x)) max(x) else NA_real_ }
  ad1 <- ad %>% group_by(pos) %>%
    summarise(gwas = safe_max(gwas), cv2f = safe_max(cv2f), incl = safe_max(incl), vcp = safe_max(vcp),
              ct = { i <- which.max(replace(vcp, is.na(vcp), -Inf)); ct[i] }, .groups = "drop")
  ev1 <- tibble(variant = cs_union, pos = vpos) %>% left_join(ad1, by = "pos")

  base_rows <- tibble(variant = cs_union,
    `SuSiE-rRSS PIP` = pipv_cs(fit_ebmix), `SuSiE-RSS PIP` = pipv_cs(fit_susie),
    `AD GWAS (-log10P)` = ev1$gwas, `cV2F` = ev1$cv2f, `xQTL inclusion` = ev1$incl) %>%
    pivot_longer(-variant, names_to = "feature", values_to = "value")
  coloc_rows <- ev1 %>% filter(!is.na(vcp)) %>% transmute(variant, feature = paste0("Coloc: ", ct), value = vcp)
  feat_order <- c("SuSiE-rRSS PIP","SuSiE-RSS PIP","AD GWAS (-log10P)","cV2F","xQTL inclusion",
                  sort(unique(coloc_rows$feature)))

  plot_long <- bind_rows(base_rows, coloc_rows) %>% group_by(feature) %>% filter(any(!is.na(value))) %>%
    mutate(fill = (value - min(value, na.rm = TRUE)) / (max(value, na.rm = TRUE) - min(value, na.rm = TRUE)),
           fill = ifelse(is.nan(fill), 0.5, fill)) %>% ungroup() %>%
    mutate(variant = factor(variant, levels = cs_union),
           feature = factor(feature, levels = rev(feat_order[feat_order %in% unique(feature)])))

  list(plot_long = plot_long,
       x_labels  = setNames(as.character(vpos), cs_union),
       title     = paste0(loc$sid, " — ", loc$bid))
}

In [ ]:
idx <- c(CR1    = 8,    # chr1_205972031_208461272  / AD_Wightman_Full_2021
         MS4A4A = 2,    # chr11_56858541_63818332   / AD_Wightman_Excluding23andMe_2021
         CASS4  = 9)    # chr20_55439357_57610823   / AD_Wightman_Full_2021

heatmaps <- map(idx, ~ locus_heatmap_data(load_fits(region_diff$block_id[.x], region_diff$study[.x], AD_loci)))
map_chr(heatmaps, "title")

save_data(heatmaps, "Exploratory.Differing_Loci_Evidence_Heatmap")